### LGBM:
1) запуск алгоритма с параметрами по умолчанию и вывод некоторой статистики
2) запуск optuna-оптимизации по части гиперпараметров
3) визуализация optuna: важность параметров и контуры
4) запуск алгоритма с найденными гиперпараметрами и вывод предварительной статистики
5) сохраняем результаты дефолтного и оптимизированного алгоритма

In [1]:
from private.utils import get_reduced_mnist_data, memory_check
from public.classification_utils import LGBM
from public.models import ClassificationProcessor

with memory_check():
    df = get_reduced_mnist_data()
    processor = ClassificationProcessor(df, "label")   
    processor.calculate({
        LGBM : {},
    })
    processor.report(LGBM)
    processor.pick_model(LGBM)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.140133 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 108649
[LightGBM] [Info] Number of data points in the train set: 56000, number of used features: 627
[LightGBM] [Info] Start training from score -2.316612
[LightGBM] [Info] Start training from score -2.184485
[LightGBM] [Info] Start training from score -2.304015
[LightGBM] [Info] Start training from score -2.282607
[LightGBM] [Info] Start training from score -2.328086
[LightGBM] [Info] Start training from score -2.405963
[LightGBM] [Info] Start training from score -2.320422
[LightGBM] [Info] Start training from score -2.261649
[LightGBM] [Info] Start training from score -2.327903
[LightGBM] [Info] Start training from score -2.308495

	light_gradient_boosting_machine


pr_auc,roc_auc,accuracy
0.996369,0.999450,0.974143


,precision,recall,f1-score,support
0,0.983,0.991,0.987,1381.000
1,0.985,0.982,0.983,1575.000
2,0.976,0.977,0.976,1398.000
3,0.964,0.968,0.966,1428.000
4,0.976,0.968,0.972,1365.000
5,0.979,0.966,0.972,1263.000
6,0.983,0.983,0.983,1375.000
7,0.976,0.975,0.975,1459.000
8,0.965,0.973,0.969,1365.000
9,0.954,0.957,0.955,1391.000


Memory Increased by: 0.88 MB


#### запуск optuna-оптимизации по части гиперпараметров

In [2]:
from public.optuna_utils import OPT_LGBM, optimize

with memory_check():
    study = optimize(
        model_type=OPT_LGBM, 
        df=df, 
        target_column="label",
        n_trials=50
    )
    print(f"Наилучшие значения гиперпараметров {study.best_params}")
    print(f"pr_auc на обучающем наборе: {study.best_value:.4f}")

[I 2026-08-17 19:19:25,980] A new study created in memory with name: Light Gradient-Boosting Machine


  0%|          | 0/50 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
Training until validation scores don't improve for 50 rounds
[10]	valid's multi_logloss: 0.575427 + 0.00306034	valid's macro_pr_auc: 0.98189 + 0.000837251
[10]	valid's multi_logloss: 0.77855 + 0.00233504	valid's macro_pr_auc: 0.979845 + 0.000682772
[10]	valid's multi_logloss: 0.581465 + 0.00307345	valid's macro_pr_auc: 0.982736

#### Визуализация optuna:
1) Сравнение важности гиперпараметров
2) Отрисовка контура оптимизации. Помогает выбрать направление дальнейшей оптимизации в сторону "темных" областей

In [3]:
from optuna.visualization import plot_param_importances

plot_param_importances(study)

In [4]:
from optuna.visualization import plot_contour

plot_contour(study)

#### Применение найденных лучших гиперпараметров:

In [5]:
with memory_check():
    alter_title = f'{LGBM}_tuned'
    processor.calculate({
        LGBM : {
            "n_estimators": 99,
            "learning_rate": 0.09948,
            "num_leaves": 81,
            'alter_title': alter_title
        },
    })
    processor.report(alter_title)
    processor.pick_model(alter_title)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.127459 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 108649
[LightGBM] [Info] Number of data points in the train set: 56000, number of used features: 627
[LightGBM] [Info] Start training from score -2.316612
[LightGBM] [Info] Start training from score -2.184485
[LightGBM] [Info] Start training from score -2.304015
[LightGBM] [Info] Start training from score -2.282607
[LightGBM] [Info] Start training from score -2.328086
[LightGBM] [Info] Start training from score -2.405963
[LightGBM] [Info] Start training from score -2.320422
[LightGBM] [Info] Start training from score -2.261649
[LightGBM] [Info] Start training from score -2.327903
[LightGBM] [Info] Start training from score -2.308495

	light_gradient_boosting_machine_tuned


pr_auc,roc_auc,accuracy
0.996779,0.999491,0.977714


,precision,recall,f1-score,support
0,0.980,0.991,0.986,1381.000
1,0.989,0.985,0.987,1575.000
2,0.976,0.974,0.975,1398.000
3,0.980,0.978,0.979,1428.000
4,0.978,0.971,0.974,1365.000
5,0.985,0.971,0.978,1263.000
6,0.981,0.984,0.983,1375.000
7,0.975,0.979,0.977,1459.000
8,0.966,0.979,0.973,1365.000
9,0.968,0.963,0.965,1391.000


Memory Increased by: 593.18 MB


#### Мини-репорт:

In [6]:
dec_tr = next((model for model in processor.models if model.title == LGBM), None)
dec_tr_tuned = next((model for model in processor.models if model.title == alter_title), None)

print(f"{LGBM} : {alter_title} >> {dec_tr.pr_auc} : {dec_tr_tuned.pr_auc}")

light_gradient_boosting_machine : light_gradient_boosting_machine_tuned >> 0.9963686913530966 : 0.996779173100143
